# Lab - System Prompt Rules vs Code Rules

Three tasks on where a rule belongs. A system prompt shapes how ShopAssist behaves; a
rule that must always hold goes in code, before any request leaves.

| Task | What you learn |
| --- | --- |
| 1 | A system prompt sets role, tone and scope - and Claude follows it |
| 2 | A rule that must always hold is a gate in code, not a line in the prompt |
| 3 | Once the condition is met, the same request passes the gate |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# Two lines for the system prompt you finish in Task 1.
SCOPE_RULE = "Help only with orders, returns and refunds. "
DECLINE_RULE = "For anything else, say politely that you cannot help with it. "

# Provided: counts every request that actually goes to Claude.
REQUESTS_SENT = 0


def ask(messages):
    global REQUESTS_SENT
    REQUESTS_SENT += 1
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=SYSTEM,
        messages=messages,
    )
    reply = None
    for block in response.content:
        if block.type == "text":
            reply = block.text
    print(reply)
    return reply


history = []

---

## Task 1 - A system prompt with role and scope

In [ ]:
# ============================================================
# TASK 1 - A system prompt with role and scope
# ============================================================
#
# WHAT TO DO
#   Finish the system prompt: a role, a scope, what to do
#   outside the scope, the return rule and short answers. Then
#   a customer asks for a poem.
#
# WHY IT MATTERS
#   The system prompt sets up Claude before the customer says
#   anything. Role, tone, scope and the style of an answer
#   follow it reliably - so the poem request is declined in a
#   friendly way, with an offer to help with orders instead.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Let's make it a real system prompt".
#
# HOW TO DO IT
#   Two blanks, both constants from the setup cell:
#
#       SCOPE_RULE     what ShopAssist helps with
#       DECLINE_RULE   what it does with everything else
# ============================================================

# TODO: replace each ... below with the constant named beside it
SYSTEM_LINES = [
    "You are ShopAssist, an online store's support assistant. ",
    ...,   # SCOPE_RULE - what ShopAssist helps with
    ...,   # DECLINE_RULE - what to do with everything else
    "Returns are accepted within 30 days of delivery. ",
    "Answer in one or two sentences.",
]

# Provided: one string out of the lines - and a plain message if a blank is left.
if ... in SYSTEM_LINES:
    raise ValueError("You still have ... in SYSTEM_LINES. Replace each one with the "
                     "constant named beside it.")
SYSTEM = "".join(SYSTEM_LINES)
print(SYSTEM)
print()


def say(text):
    history.append({"role": "user", "content": text})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
    return reply


poem_reply = say("Write me a short poem about jackets.")

check("scope", system=SYSTEM, poem_reply=poem_reply)

---

## Task 2 - A rule that must hold lives in code

In [ ]:
# ============================================================
# TASK 2 - A rule that must hold lives in code
# ============================================================
#
# WHAT TO DO
#   Add a gate to say(): a refund request from a customer who
#   has not confirmed the email of the order is answered by our
#   code, and never sent to Claude.
#
# WHY IT MATTERS
#   It is tempting to add this rule as one more line in the
#   system prompt. But a prompt is a request, not a guarantee:
#   a customer who insists, or claims the check was already
#   done, may talk the model into it. A rule that must always
#   hold belongs in code, where no conversation can change it.
#
#   Watch the count of requests sent: it does not move for the
#   refund message. Our code decided alone.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now a different kind of rule".
#
# HOW TO DO IT
#   Two blanks in the gate's condition:
#
#       "refund"    the word the gate looks for, in quotes
#       verified    the flag - the gate fires when it is NOT set
# ============================================================

verified = False


# TODO: replace each ... below with the value named beside it
def say(text):
    if ... in text.lower() and not ...:   # "refund", and the verified flag
        print("Please confirm the email of your order first.")
        return None
    history.append({"role": "user", "content": text})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
    return reply


sent_before = REQUESTS_SENT
try:
    say("I want a refund for the jacket I bought 20 days ago.")
except TypeError as exc:   # provided: a blank left in the gate, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... in the gate. Replace each one with the "
                         "value named beside it.") from None
    raise
sent_after = REQUESTS_SENT
print("requests sent to Claude for this message:", sent_after - sent_before)

check("gate", sent_before=sent_before, sent_after=sent_after, say=say)

---

## Task 3 - Verified, the same request passes

In [ ]:
# ============================================================
# TASK 3 - Verified, the same request passes
# ============================================================
#
# WHAT TO DO
#   The customer confirms the email. Mark them verified and send
#   the same refund message again.
#
# WHY IT MATTERS
#   In a real system the application would check the email
#   against the order. Here we set the flag. This time the
#   message passes the gate, reaches Claude, and Claude answers
#   about the 30-day window. Three cases, three outcomes: the
#   poem declined by the prompt, the first refund stopped by
#   code, the second one allowed through.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then the customer confirms the email".
#
# HOW TO DO IT
#   One blank: the value that marks the customer as confirmed.
#
#       True    with a capital T, no quotes
# ============================================================

# TODO: replace the ... below with the value named beside it
verified = ...   # True - the customer confirmed the email

sent_before = REQUESTS_SENT
try:
    refund_reply = say("I want a refund for the jacket I bought 20 days ago.")
except TypeError as exc:   # provided: a blank still left in Task 2's gate
    if "ellipsis" in str(exc):
        raise ValueError("Task 2's gate still has ... in it. Finish Task 2, run it, "
                         "then run this cell.") from None
    raise
sent_after = REQUESTS_SENT
print("requests sent to Claude for this message:", sent_after - sent_before)

check("verified", verified=verified, sent_before=sent_before, sent_after=sent_after,
      refund_reply=refund_reply)

---

## Done

Put role, tone and scope in the system prompt - Claude follows them reliably. Put every
rule that must always hold in code, before the request leaves your application. The
prompt is guidance; the gate is a guarantee.